# Stage 1 Baseline - SIF Reverse Dictionary

This notebook builds the first baseline from the proposal: **SIF**, or Smooth Inverse Frequency sentence embeddings.

The basic idea is simple: represent each definition using the words inside it, reduce the effect of very common words, and compare that definition vector with candidate word vectors.

## Step 1 - Load preprocessed data

Use the cleaned training file created in the preprocessing notebook.

In [1]:
from pathlib import Path
from collections import Counter
import re

import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA_PATH = Path("..") / "data" / "processed" / "opted_train.csv"
RESULTS_DIR = Path("..") / "results" / "baselines"

if not DATA_PATH.exists():
    DATA_PATH = Path("data") / "processed" / "opted_train.csv"
    RESULTS_DIR = Path("results") / "baselines"

df = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False)

print(f"Loaded training rows: {len(df):,}")
display(df.head())

Loaded training rows: 140,406


,entry_id,split,word_original,word_norm,sense_number,definition_original,definition_norm,definition_basic_clean,definition_word_count,clean_definition_word_count,is_short_definition,is_long_definition,count_original,pos_original
0,4a92354c44a6,train,'Em,'em,1,An obsolete or colloquial contraction of the o...,an obsolete or colloquial contraction of the o...,an obsolete or colloquial contraction of the o...,11,11,False,False,3,
1,e6a183d531df,train,'Gainst,'gainst,1,A contraction of Against.,a contraction of against.,a contraction of against,4,4,False,False,7,prep.
2,9fc15aeb2e7a,train,'Neath,'neath,1,An abbreviation of Beneath.,an abbreviation of beneath.,an abbreviation of beneath,4,4,False,False,6,prep. & adv.
3,dfba73470c2c,train,'s,'s,1,A contraction for is or (colloquially) for has.,a contraction for is or (colloquially) for has.,a contraction for is or colloquially for has,8,8,False,False,2,
4,73b26104ffe8,train,'Sdeath,'sdeath,1,An exclamation expressive of impatience or anger.,an exclamation expressive of impatience or anger.,an exclamation expressive of impatience or anger,7,7,False,False,7,interj.


## Step 2 - Take a 10% subset

Stage 1 in the proposal uses 10% of the data, so this notebook follows that setup.

In [2]:
RANDOM_SEED = 42
SUBSET_FRAC = 0.10

subset_df = (
    df.sample(frac=SUBSET_FRAC, random_state=RANDOM_SEED)
    .reset_index(drop=True)
    .copy()
)

print(f"Subset rows: {len(subset_df):,}")
print(f"Subset unique words: {subset_df['word_norm'].nunique():,}")
display(subset_df[["entry_id", "word_original", "definition_original"]].head())

Subset rows: 14,041
Subset unique words: 12,526


,entry_id,word_original,definition_original
0,9ba728da1dea,Attribution,That which is ascribed or attributed.
1,ce7c005bcbf0,Chinoidine,See Quinodine.
2,3272fa69b930,Cylindroid,A solid body resembling a right cylinder but h...
3,cec643507a7a,Compel,To force to yield; to overpower; to subjugate.
4,c4e2b1fe71b4,Quab,See Quob v. i.


## Step 3 - Remove cross-reference-only definitions

Some OPTED entries are pointers, not definitions. Examples: `See Quinodine.` or `Same as ...`. For this baseline, those rows are removed because SIF cannot follow dictionary links.

In [ ]:
initial_subset_rows = len(subset_df)

CROSS_REFERENCE_PATTERN = re.compile(
    r"^\s*(see|same as|alt\.?\s+of|alternative form of|another form of)\b",
    flags=re.IGNORECASE,
)

subset_df["is_cross_reference_definition"] = subset_df["definition_original"].map(
    lambda text: bool(CROSS_REFERENCE_PATTERN.search(str(text)))
)

cross_reference_examples = subset_df.loc[
    subset_df["is_cross_reference_definition"],
    ["word_original", "definition_original"],
].head(10)

cross_reference_rows_removed = int(subset_df["is_cross_reference_definition"].sum())

print(f"Cross-reference rows removed: {cross_reference_rows_removed:,}")
display(cross_reference_examples)

subset_df = (
    subset_df.loc[~subset_df["is_cross_reference_definition"]]
    .drop(columns=["is_cross_reference_definition"])
    .reset_index(drop=True)
    .copy()
)

print(f"Rows remaining for SIF baseline: {len(subset_df):,}")

## Step 4 - Tokenize definitions and candidate words

Split definitions and words into lowercase tokens before building vectors.

In [3]:
TOKEN_PATTERN = re.compile(r"\b[a-z]+(?:'[a-z]+)?\b")

def tokenize(text):
    return TOKEN_PATTERN.findall(str(text).lower())


subset_df["definition_tokens"] = subset_df["definition_basic_clean"].map(tokenize)
subset_df["target_tokens"] = subset_df["word_norm"].map(tokenize)

display(subset_df[["word_original", "definition_tokens", "target_tokens"]].head())

,word_original,definition_tokens,target_tokens
0,Attribution,"[that, which, is, ascribed, or, attributed]",[attribution]
1,Chinoidine,"[see, quinodine]",[chinoidine]
2,Cylindroid,"[a, solid, body, resembling, a, right, cylinde...",[cylindroid]
3,Compel,"[to, force, to, yield, to, overpower, to, subj...",[compel]
4,Quab,"[see, quob, v, i]",[quab]


## Step 5 - Learn lightweight word embeddings from the subset

SIF needs word vectors. Since this repo does not include GloVe or word2vec files, this notebook creates small local vectors from the OPTED subset using SVD.

In [4]:
EMBEDDING_DIM = 100
MAX_FEATURES = 30000

embedding_corpus = pd.concat(
    [
        subset_df["definition_basic_clean"],
        subset_df["word_norm"],
    ],
    ignore_index=True,
)

vectorizer = CountVectorizer(
    tokenizer=tokenize,
    token_pattern=None,
    lowercase=False,
    max_features=MAX_FEATURES,
    min_df=1,
)

term_matrix = vectorizer.fit_transform(embedding_corpus)
actual_dim = min(EMBEDDING_DIM, term_matrix.shape[1] - 1)

svd = TruncatedSVD(n_components=actual_dim, random_state=RANDOM_SEED)
svd.fit(term_matrix)

vocab = vectorizer.vocabulary_
terms = vectorizer.get_feature_names_out()
term_embeddings = (svd.components_.T * svd.singular_values_).astype(np.float32)

print(f"Vocabulary size: {len(vocab):,}")
print(f"Embedding dimension: {actual_dim}")

Vocabulary size: 29,404
Embedding dimension: 100


## Step 6 - Compute SIF weights

Common words get smaller weights. More specific words usually carry more meaning.

In [ ]:
SIF_A = 1e-3

definition_token_counts = Counter(
    token
    for tokens in subset_df["definition_tokens"]
    for token in tokens
)
total_definition_tokens = sum(definition_token_counts.values())

def sif_weight(token):
    frequency = definition_token_counts.get(token, 1)
    probability = frequency / max(total_definition_tokens, 1)
    return SIF_A / (SIF_A + probability)


print(f"Total definition tokens used for SIF weights: {total_definition_tokens:,}")
print(f"Example weight for 'the': {sif_weight('the'):.4f}")
print(f"Example weight for 'dictionary': {sif_weight('dictionary'):.4f}")

## Step 7 - Encode text using SIF

Create one vector per definition and one vector per candidate word.

In [ ]:
def encode_tokens(tokens):
    weighted_vectors = []

    for token in tokens:
        token_index = vocab.get(token)
        if token_index is None:
            continue

        weighted_vectors.append(term_embeddings[token_index] * sif_weight(token))

    if not weighted_vectors:
        return np.zeros(actual_dim, dtype=np.float32)

    return np.mean(weighted_vectors, axis=0).astype(np.float32)


definition_embeddings = np.vstack(
    subset_df["definition_tokens"].map(encode_tokens).to_numpy()
)

candidate_words = (
    subset_df[["word_original", "word_norm", "target_tokens"]]
    .drop_duplicates(subset=["word_norm"])
    .reset_index(drop=True)
)

candidate_embeddings = np.vstack(
    candidate_words["target_tokens"].map(encode_tokens).to_numpy()
)

print(f"Definition embedding matrix: {definition_embeddings.shape}")
print(f"Candidate word embedding matrix: {candidate_embeddings.shape}")

## Step 8 - Remove the common direction

SIF removes the strongest shared component from the vectors. This reduces generic background noise.

In [ ]:
nonzero_definition_mask = np.linalg.norm(definition_embeddings, axis=1) > 0
nonzero_candidate_mask = np.linalg.norm(candidate_embeddings, axis=1) > 0

pc_training_matrix = np.vstack(
    [
        definition_embeddings[nonzero_definition_mask],
        candidate_embeddings[nonzero_candidate_mask],
    ]
)

pc_svd = TruncatedSVD(n_components=1, random_state=RANDOM_SEED)
pc_svd.fit(pc_training_matrix)
common_direction = pc_svd.components_[0]

def remove_common_direction(matrix):
    return matrix - matrix.dot(common_direction.reshape(-1, 1)) * common_direction


definition_embeddings = remove_common_direction(definition_embeddings).astype(np.float32)
candidate_embeddings = remove_common_direction(candidate_embeddings).astype(np.float32)

print("Removed first common component from SIF embeddings.")

## Step 9 - Prepare evaluable examples

Keep only rows where both the definition and the correct target word have usable vectors.

In [ ]:
candidate_words = candidate_words.loc[nonzero_candidate_mask].reset_index(drop=True)
candidate_embeddings = candidate_embeddings[nonzero_candidate_mask]

candidate_index = {
    word: index
    for index, word in enumerate(candidate_words["word_norm"])
}

eval_mask = (
    nonzero_definition_mask
    & subset_df["word_norm"].isin(candidate_index)
)

eval_df = subset_df.loc[eval_mask].reset_index(drop=True)
eval_embeddings = definition_embeddings[eval_mask]
target_indices = eval_df["word_norm"].map(candidate_index).to_numpy()

print(f"Candidate words with usable vectors: {len(candidate_words):,}")
print(f"Evaluable query definitions: {len(eval_df):,}")
print(f"Dropped query definitions: {len(subset_df) - len(eval_df):,}")

## Step 10 - Rank candidate words

For each definition, rank all candidate words by cosine similarity and record where the correct word appears.

In [ ]:
def l2_normalize(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return matrix / norms


eval_embeddings_norm = l2_normalize(eval_embeddings)
candidate_embeddings_norm = l2_normalize(candidate_embeddings)

BATCH_SIZE = 256
ranks = []
top_predictions = []

for start in range(0, len(eval_df), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(eval_df))
    batch_embeddings = eval_embeddings_norm[start:end]
    batch_targets = target_indices[start:end]
    similarities = batch_embeddings @ candidate_embeddings_norm.T

    for row_offset, target_index in enumerate(batch_targets):
        row_scores = similarities[row_offset]
        target_score = row_scores[target_index]
        rank = int(np.sum(row_scores > target_score) + 1)
        ranks.append(rank)

        if len(top_predictions) < 20:
            top_index = int(np.argmax(row_scores))
            query_row = eval_df.iloc[start + row_offset]
            top_predictions.append(
                {
                    "query_word": query_row["word_original"],
                    "query_definition": query_row["definition_original"],
                    "predicted_word": candidate_words.iloc[top_index]["word_original"],
                    "correct_rank": rank,
                    "top_score": float(row_scores[top_index]),
                    "target_score": float(target_score),
                }
            )

ranks = np.array(ranks)

print(f"Computed ranks for {len(ranks):,} examples.")

## Step 11 - Calculate Stage 1 metrics

Calculate Accuracy@1, Accuracy@10, Accuracy@100, and Median Rank.

In [ ]:
metrics = {
    "model": "SIF_local_svd",
    "subset_fraction": SUBSET_FRAC,
    "num_initial_subset_rows": initial_subset_rows,
    "num_cross_reference_rows_removed": cross_reference_rows_removed,
    "num_subset_rows": len(subset_df),
    "num_evaluable_queries": len(eval_df),
    "num_candidate_words": len(candidate_words),
    "accuracy_at_1": float(np.mean(ranks <= 1)),
    "accuracy_at_10": float(np.mean(ranks <= 10)),
    "accuracy_at_100": float(np.mean(ranks <= 100)),
    "median_rank": float(np.median(ranks)),
}

metrics_df = pd.DataFrame([metrics])
display(metrics_df)

## Step 12 - Inspect sample predictions

Look at a few predictions to understand what the baseline is getting right or wrong.

In [ ]:
sample_predictions_df = pd.DataFrame(top_predictions)
display(sample_predictions_df)

## Step 13 - Save baseline results

Save metrics and sample predictions so later models can be compared using the same format.

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

metrics_df.to_csv(RESULTS_DIR / "sif_metrics.csv", index=False)
sample_predictions_df.to_csv(RESULTS_DIR / "sif_sample_predictions.csv", index=False)

print(f"Saved SIF metrics to: {RESULTS_DIR / 'sif_metrics.csv'}")
print(f"Saved SIF sample predictions to: {RESULTS_DIR / 'sif_sample_predictions.csv'}")

## Baseline notes

For Stage 1, we use SIF as the first baseline for the reverse dictionary task. This gives us a simple retrieval result before moving to BiLSTM, BERT, and the final JEPA-style model.

Each dictionary definition is converted into a fixed-size vector by averaging word vectors with SIF weights. Common words get smaller weights, and more specific words get larger weights. The final definition vector is compared with candidate word vectors using cosine similarity.

Pretrained GloVe or word2vec files are not included in this repo, so local word vectors are created from the OPTED subset using CountVectorizer and TruncatedSVD. If pretrained vectors are added later, this notebook can be updated.

This run uses 10% of the preprocessed training data. Cross-reference-only definitions such as `See ...`, `Same as ...`, and `Alt. of ...` are removed because they do not describe the target word directly. The remaining definitions are used as queries, and all unique words in the same subset are used as candidate answers.

Evaluation is done as a ranking task. For every definition, candidate words are sorted by cosine similarity. Accuracy@1, Accuracy@10, and Accuracy@100 check whether the correct word appears in the top 1, 10, or 100 results. Median Rank shows the typical position of the correct word.

This baseline does not use JEPA. JEPA comes later, when we train a predictor from definition embeddings to target word or word-sense embeddings. SIF is kept as the simple comparison point.